# Challenge 1 - T-test

In statistics, t-test is used to test if two data samples have a significant difference between their means. There are two types of t-test:

* **Student's t-test** (a.k.a. independent or uncorrelated t-test). This type of t-test is to compare the samples of **two independent populations** (e.g. test scores of students in two different classes). `scipy` provides the [`ttest_ind`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) method to conduct student's t-test.

* **Paired t-test** (a.k.a. dependent or correlated t-test). This type of t-test is to compare the samples of **the same population** (e.g. scores of different tests of students in the same class). `scipy` provides the [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html) method to conduct paired t-test.

Both types of t-tests return a **p-value**: the probability of seeing a difference at least this large if the null hypothesis (no difference) were true. Choose the significance level before running the test, usually α = 0.05. If the p-value is below α, reject the null hypothesis: the difference is statistically significant. Otherwise, fail to reject it. A p-value of 0.07 is not "almost significant"; it is a fail to reject at α = 0.05.

For two independent groups, use Welch's version, `ttest_ind(a, b, equal_var=False)`: it does not assume the two groups have the same spread, so it is the safe default.

Read more about the t-test in [this article](https://researchbasics.education.uconn.edu/t-test/) and [this Quora](https://www.quora.com/What-is-the-difference-between-a-paired-and-unpaired-t-test). Make sure you understand when to use which type of t-test. 

In [1]:
# Import libraries

import pandas as pd
from scipy.stats import ttest_ind, ttest_rel

#### Import dataset

In this challenge we will work on the Pokemon dataset in the `data` folder of this repository. The goal is to test whether different groups of pokemon (e.g. Legendary vs Normal, Generation 1 vs 2, single-type vs dual-type) have different stats (e.g. HP, Attack, Defense, etc.).

In [2]:
# Import dataset

pokemon = pd.read_csv('../data/Pokemon.csv')

pokemon.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


#### First we want to define a function with which we can test the means of a feature set of two samples. 

In the next cell you'll see the annotations of the Python function that explains what this function does and its arguments and returned value. This type of annotation is called **docstring** which is a convention used among Python developers. The docstring convention allows developers to write consistent tech documentations for their codes so that others can read. It also allows some websites to automatically parse the docstrings and display user-friendly documentations.

Follow the specifications of the docstring and complete the function.

In [3]:
def t_test_features(s1, s2, features=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Total']):
    """Test means of a feature set of two samples
    
    Args:
        s1 (dataframe): sample 1
        s2 (dataframe): sample 2
        features (list): an array of features to test
    
    Returns:
        dict: a dictionary of t-test scores for each feature where the feature name is the key and the p-value is the value
    """
    results = {}

    # Welch's t-test, because it does not assume that both groups have the same spread
    for feature in features:
        results[feature] = float(ttest_ind(s1[feature], s2[feature], equal_var=False).pvalue)
    
    return results

#### Using the `t_test_features` function, conduct t-test for Lengendary vs non-Legendary pokemons.

*Hint: use Welch's t-test (`equal_var=False`). Your output should look like below:*

```
{'HP': 1.0026911708035284e-13,
 'Attack': 2.520372449236646e-16,
 'Defense': 4.8269984949193316e-11,
 'Sp. Atk': 1.5514614112239812e-21,
 'Sp. Def': 2.2949327864052826e-15,
 'Speed': 1.049016311882451e-18,
 'Total': 9.357954335957446e-47}
 ```

First the hypotheses, at a significance level of α = 0.05:

> **H0**: for each feature, the mean of Legendary pokemons and the mean of non-Legendary pokemons are the same.

> **H1**: for each feature, the mean of Legendary pokemons and the mean of non-Legendary pokemons are different.

In [4]:
legendary = pokemon[pokemon['Legendary']]
non_legendary = pokemon[~pokemon['Legendary']]

legendary_results = t_test_features(legendary, non_legendary)
legendary_results

{'HP': 1.0026911708035284e-13,
 'Attack': 2.5203724492366553e-16,
 'Defense': 4.8269984949193316e-11,
 'Sp. Atk': 1.5514614112239705e-21,
 'Sp. Def': 2.294932786405291e-15,
 'Speed': 1.0490163118824585e-18,
 'Total': 9.357954335957446e-47}

In [5]:
# what are the means actually? They show the direction of the differences
pd.DataFrame({'Legendary': legendary[list(legendary_results)].mean(), 'Non-Legendary': non_legendary[list(legendary_results)].mean()}).round(2)

,Legendary,Non-Legendary
HP,92.74,67.18
Attack,116.68,75.67
Defense,99.66,71.56
Sp. Atk,122.18,68.45
Sp. Def,105.94,68.89
Speed,100.18,65.46
Total,637.38,417.21


#### From the test results above, what conclusion can you make? Do Legendary and non-Legendary pokemons have significantly different stats on each feature?

**Answer:** All 7 p-values are far below *0.05* (alpha), so we can reject the Null-Hypothesis for every feature. Legendary and non-Legendary pokemons have significantly different stats on each feature. The means show the direction: Legendary pokemons are higher on all of them. The biggest difference of the single stats is `Sp. Atk` (about 122.2 vs 68.5) and `Total` is about 637.4 vs 417.2.

#### Next, conduct t-test for Generation 1 and Generation 2 pokemons.

First the hypotheses, at a significance level of α = 0.05:

> **H0**: for each feature, the mean of Generation 1 and the mean of Generation 2 are the same.

> **H1**: for each feature, the mean of Generation 1 and the mean of Generation 2 are different.

In [6]:
generation_1 = pokemon[pokemon['Generation'] == 1]
generation_2 = pokemon[pokemon['Generation'] == 2]

generation_results = t_test_features(generation_1, generation_2)
generation_results

{'HP': 0.14551697834219632,
 'Attack': 0.24721958967217725,
 'Defense': 0.5677711011725426,
 'Sp. Atk': 0.12332165977104392,
 'Sp. Def': 0.18829872292645758,
 'Speed': 0.0023926593731213508,
 'Total': 0.5631377907941676}

In [7]:
# what are the means actually? They show the direction of the differences
pd.DataFrame({'Generation 1': generation_1[list(generation_results)].mean(), 'Generation 2': generation_2[list(generation_results)].mean()}).round(2)

,Generation 1,Generation 2
HP,65.82,71.21
Attack,76.64,72.03
Defense,70.86,73.39
Sp. Atk,71.82,65.94
Sp. Def,69.09,73.91
Speed,72.58,61.81
Total,426.81,418.28


#### What conclusions can you make?

**Answer:** Only `Speed` has a p-value below *0.05* (alpha), which is about 0.0024. So there we can reject the Null-Hypothesis: Generation 1 pokemons are faster on average (about 72.6 vs 61.8). For the other 6 features the p-values are between 0.12 and 0.57, so we fail to reject it. That does not prove that those means are the same, the data just doesn't show a significant difference. As it seems neither generation is stronger overall (`Total` is about 426.8 vs 418.3).

#### Compare pokemons who have single type vs those having two types.

First the hypotheses, at a significance level of α = 0.05:

> **H0**: for each feature, the mean of pokemons with a single type and the mean of pokemons with two types are the same.

> **H1**: for each feature, the mean of pokemons with a single type and the mean of pokemons with two types are different.

In [8]:
# single type pokemons have no value in 'Type 2'
single_type = pokemon[pokemon['Type 2'].isna()]
two_types = pokemon[pokemon['Type 2'].notna()]

type_results = t_test_features(single_type, two_types)
type_results

{'HP': 0.11314389855379417,
 'Attack': 0.00014932578145948305,
 'Defense': 2.797854041151469e-08,
 'Sp. Atk': 0.00013876216585667901,
 'Sp. Def': 0.00010730610934512777,
 'Speed': 0.024217032818190928,
 'Total': 1.1157056505229961e-07}

In [9]:
# what are the means actually? They show the direction of the differences
pd.DataFrame({'Single type': single_type[list(type_results)].mean(), 'Two types': two_types[list(type_results)].mean()}).round(2)

,Single type,Two types
HP,67.77,70.65
Attack,74.53,83.17
Defense,67.59,79.68
Sp. Atk,68.28,77.05
Sp. Def,67.97,75.57
Speed,65.88,70.51
Total,412.02,456.63


#### What conclusions can you make?

**Answer:** Except for `HP` (p-value about 0.11), all p-values are below *0.05* (alpha), so we can reject the Null-Hypothesis for those features and fail to reject it for `HP`. Pokemons with two types are higher on all of them. The biggest difference of the single stats is `Defense` (about 79.7 vs 67.6) and `Total` is about 456.6 vs 412.0.

The weakest result is `Speed` (p-value about 0.024). With 7 tests per comparison the chance of a false positive piles up, like the lesson says about ANOVA. With a stricter level of 0.05 / 7 (about 0.007) `Speed` would not be significant anymore, while `Attack`, `Defense`, `Sp. Atk`, `Sp. Def` and `Total` would stay significant. The `Speed` result of Generation 1 vs 2 also holds at that level.

#### Now, we want to compare whether there are significant differences of `Attack` vs `Defense`  and  `Sp. Atk` vs `Sp. Def` of all pokemons. Please write your code below.

*Hint: are you comparing different populations or the same population?*

Here we compare two columns of the same pokemons, so every `Attack` belongs to a `Defense`. That makes the samples dependent and the fitting test is the paired t-test.

> **H0**: the mean of `Attack` and the mean of `Defense` are the same (and the same for `Sp. Atk` and `Sp. Def`).

> **H1**: the mean of `Attack` and the mean of `Defense` are different (and the same for `Sp. Atk` and `Sp. Def`).

In [10]:
# every pokemon has both values, so the two columns are the same population: paired t-test
print("Attack vs Defense:", ttest_rel(pokemon['Attack'], pokemon['Defense']))
print("Sp. Atk vs Sp. Def:", ttest_rel(pokemon['Sp. Atk'], pokemon['Sp. Def']))

pokemon[['Attack', 'Defense', 'Sp. Atk', 'Sp. Def']].mean().round(2)

Attack vs Defense: TtestResult(statistic=np.float64(4.325566393330478), pvalue=np.float64(1.7140303479358558e-05), df=np.int64(799))
Sp. Atk vs Sp. Def: TtestResult(statistic=np.float64(0.853986188453353), pvalue=np.float64(0.3933685997548122), df=np.int64(799))


Attack     79.00
Defense    73.84
Sp. Atk    72.82
Sp. Def    71.90
dtype: float64

#### What conclusions can you make?

**Answer:** For `Attack` vs `Defense` the p-value is about 0.000017, which is below *0.05* (alpha), so we can reject the Null-Hypothesis. On average pokemons have a higher `Attack` than `Defense` (about 79.0 vs 73.8). For `Sp. Atk` vs `Sp. Def` the p-value is about 0.39, so we fail to reject the Null-Hypothesis. There is no significant difference there (about 72.8 vs 71.9), which does not prove that they are the same.

Some limitations for all the tests in this notebook:

1. The dataset seems to contain all pokemons of these generations, so it isn't really a sample. The tests only make sense if we treat it as a sample of a bigger population.
2. 79 rows share their `#` with another row because they are different forms of the same pokemon (for example Mega forms), so not all rows are fully independent.
3. `Total` is the sum of the 6 other features, so it is not a separate piece of evidence.